# 📓 Notebook 02 – Model Implementation
**Version 1 | Road Damage Detection**

This notebook covers:
1. MobileNetV2 – transfer learning setup
2. Custom CNN – built from scratch
3. YOLOv8n – detection model setup
4. Parameter counts and output shape verification
5. YOLO dataset config for training

In [4]:
import sys
from pathlib import Path
import torch

sys.path.insert(0, str(Path('..') / 'src'))
from models import get_model, build_mobilenetv2, CustomCNN, count_parameters, NUM_CLASSES

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Classes: {NUM_CLASSES}  → D00, D10, D20, D40, Normal')

Device : cpu
Classes: 5  → D00, D10, D20, D40, Normal


## 1. MobileNetV2 – Transfer Learning

- Backbone pretrained on ImageNet (frozen by default)
- Only the custom classifier head is trained
- Input: `(B, 3, 224, 224)` → Output: `(B, 5)`

```
MobileNetV2 Backbone (frozen)
  19 Inverted Residual Blocks
  Output feature map: 1280-dim vector
        ↓
Custom Head:
  Dropout(0.3) → Linear(1280→256) → ReLU → Dropout(0.2) → Linear(256→5)
        ↓
  5-class logits
```

In [5]:
mobilenet = build_mobilenetv2(num_classes=NUM_CLASSES, freeze_backbone=True)

total_params    = sum(p.numel() for p in mobilenet.parameters())
trainable_params = count_parameters(mobilenet)
frozen_params   = total_params - trainable_params

print('── MobileNetV2 ──────────────────────────────')
print(f'  Total params    : {total_params:,}')
print(f'  Trainable params: {trainable_params:,}  ← only the head')
print(f'  Frozen params   : {frozen_params:,}  ← backbone')

# Verify forward pass
dummy = torch.randn(2, 3, 224, 224)
out   = mobilenet(dummy)
print(f'  Output shape    : {out.shape}  ← (batch=2, classes=5)')

── MobileNetV2 ──────────────────────────────
  Total params    : 2,553,093
  Trainable params: 329,221  ← only the head
  Frozen params   : 2,223,872  ← backbone
  Output shape    : torch.Size([2, 5])  ← (batch=2, classes=5)


In [6]:
# Inspect the classifier head
print('Classifier head:')
print(mobilenet.classifier)

Classifier head:
Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=1280, out_features=256, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.2, inplace=False)
  (4): Linear(in_features=256, out_features=5, bias=True)
)


## 2. Custom CNN – Built from Scratch

- No pretrained weights — learns entirely from RDD2022
- 4 conv blocks with BatchNorm + MaxPool
- Input: `(B, 3, 224, 224)` → Output: `(B, 5)`

```
Input 224×224
  Block 1: Conv(3→32)   + BN + ReLU + MaxPool → 112×112
  Block 2: Conv(32→64)  + BN + ReLU + MaxPool →  56×56
  Block 3: Conv(64→128) + BN + ReLU + MaxPool →  28×28
  Block 4: Conv(128→256)+ BN + ReLU + MaxPool →  14×14
  AdaptiveAvgPool(4×4)                         →   4×4
        ↓
  Flatten → Linear(4096→512) → ReLU → Dropout(0.4) → Linear(512→5)
```

In [7]:
cnn = CustomCNN(num_classes=NUM_CLASSES)

print('── Custom CNN ───────────────────────────────')
print(f'  Trainable params: {count_parameters(cnn):,}  ← all params trained from scratch')

dummy = torch.randn(2, 3, 224, 224)
out   = cnn(dummy)
print(f'  Output shape    : {out.shape}')

# Show feature map sizes at each block
print('\nFeature map sizes through each block:')
x = dummy
for i, block in enumerate(cnn.features):
    x = block(x)
    print(f'  After block {i+1}: {tuple(x.shape)}')

── Custom CNN ───────────────────────────────
  Trainable params: 2,489,125  ← all params trained from scratch
  Output shape    : torch.Size([2, 5])

Feature map sizes through each block:
  After block 1: (2, 32, 112, 112)
  After block 2: (2, 64, 56, 56)
  After block 3: (2, 128, 28, 28)
  After block 4: (2, 256, 14, 14)
  After block 5: (2, 256, 4, 4)


## 3. YOLOv8n – Detection Model

- Pretrained on COCO, fine-tuned on RDD2022
- Outputs bounding boxes + class labels + confidence
- Requires YOLO-format dataset (images + `.txt` label files)

```
Input: 640×640
  CSPDarknet Backbone → PANet Neck → Detection Head
  Output: [x_center, y_center, width, height, confidence, class_probs]
```

In [8]:
from ultralytics import YOLO

yolo = YOLO('yolov8n.pt')  # downloads ~6MB pretrained weights

print('── YOLOv8n ──────────────────────────────────')
print(f'  Task  : {yolo.task}')
print(f'  Model : {type(yolo.model).__name__}')

total = sum(p.numel() for p in yolo.model.parameters())
print(f'  Params: {total:,}')

── YOLOv8n ──────────────────────────────────
  Task  : detect
  Model : DetectionModel
  Params: 3,157,200


## 4. YOLO Dataset Config

YOLOv8 needs a `dataset.yaml` file pointing to images and label files.
We generate it here so notebook 03 can use it directly for training.

In [9]:
import yaml
from pathlib import Path

DATA_ROOT = Path('../data').resolve()
YOLO_DIR  = DATA_ROOT / 'yolo'

# Create YOLO folder structure
for split in ['train', 'val']:
    (YOLO_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

# Write dataset.yaml
yaml_content = {
    'path' : str(YOLO_DIR),
    'train': 'train/images',
    'val'  : 'val/images',
    'nc'   : 4,
    'names': ['D00', 'D10', 'D20', 'D40']
}

yaml_path = YOLO_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print(f'✓ YOLO dataset config written → {yaml_path}')
print(f'  Populate {YOLO_DIR}/train/ and val/ with images + labels before training.')

✓ YOLO dataset config written → C:\Users\Srusti\Downloads\eclipse-java-2024-09-R-win32-x86_64\project\full stack\RoadDamageDetection_GITHUB\RoadDamageDetection\v1\data\yolo\dataset.yaml
  Populate C:\Users\Srusti\Downloads\eclipse-java-2024-09-R-win32-x86_64\project\full stack\RoadDamageDetection_GITHUB\RoadDamageDetection\v1\data\yolo/train/ and val/ with images + labels before training.


## 5. Model Comparison Summary

In [10]:
import pandas as pd

yolo_params = sum(p.numel() for p in yolo.model.parameters())

summary = pd.DataFrame([
    {'Model'      : 'YOLOv8n',
     'Type'       : 'Detection',
     'Params'     : f'{yolo_params:,}',
     'Input'      : '640×640',
     'Pretrained' : 'COCO',
     'Output'     : 'BBoxes + Labels'},
    {'Model'      : 'MobileNetV2',
     'Type'       : 'Classifier',
     'Params'     : f'{count_parameters(mobilenet):,}  (head only)',
     'Input'      : '224×224',
     'Pretrained' : 'ImageNet',
     'Output'     : '5-class probs'},
    {'Model'      : 'Custom CNN',
     'Type'       : 'Classifier',
     'Params'     : f'{count_parameters(cnn):,}  (from scratch)',
     'Input'      : '224×224',
     'Pretrained' : 'None',
     'Output'     : '5-class probs'},
])

print(summary.to_string(index=False))
print('\n✅ All 3 models verified. Open notebook 03 to train and evaluate.')

      Model       Type                    Params   Input Pretrained          Output
    YOLOv8n  Detection                 3,157,200 640×640       COCO BBoxes + Labels
MobileNetV2 Classifier      329,221  (head only) 224×224   ImageNet   5-class probs
 Custom CNN Classifier 2,489,125  (from scratch) 224×224       None   5-class probs

✅ All 3 models verified. Open notebook 03 to train and evaluate.


## 6. Model Summaries (torchinfo)

Run this cell after all models are built. It prints layer-by-layer breakdowns for all three models.

In [11]:
import sys, torch
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))
from models import build_mobilenetv2, CustomCNN, NUM_CLASSES
from ultralytics import YOLO

! pip install -q torchinfo
from torchinfo import summary

mobilenet = build_mobilenetv2(num_classes=NUM_CLASSES, freeze_backbone=True)
cnn       = CustomCNN(num_classes=NUM_CLASSES)
yolo      = YOLO('yolov8n.pt')

print('── MobileNetV2 Summary ──────────────────────────────')
summary(mobilenet, input_size=(1, 3, 224, 224), device='cpu')

print('\n── Custom CNN Summary ───────────────────────────────')
summary(cnn, input_size=(1, 3, 224, 224), device='cpu')

print('\n── YOLOv8n Summary ──────────────────────────────────')
yolo.info(verbose=True)

── MobileNetV2 Summary ──────────────────────────────

── Custom CNN Summary ───────────────────────────────

── YOLOv8n Summary ──────────────────────────────────
YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs


(129, 3157200, 0, 8.8575488)